In [5]:
#Imports and Paths
# Same imports, path definitions, model architecture definition and weight
# loading as Dissertation2.ipynb. Identical logic applies here as well just like Diss2.ipynb
# The key difference is here I'm scoring NEW images (GM Street View),
# not evaluating on Place Pulse test pairs.
import torch
import torch.nn as nn
import torchvision.models as models
import torchvision.transforms as transforms
from torch.utils.data import Dataset, DataLoader
import pandas as pd
import numpy as np
from PIL import Image
import os
from tqdm import tqdm

#Defining my paths
base       = '.\\data'
IMG_2012   = f'{base}\\Raw\\Streetview_2012'
IMG_2019   = f'{base}\\Raw\\Streetview_2019'
MODEL_PATH = f'{base}\\Outputs\\Models\\best_model.pth'
SCORES_OUT = f'{base}\\Outputs\\Scores\\image_scores.csv'

os.makedirs(f'{base}\\Outputs\\Scores',exist_ok=True)
device=torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print("Using:", device)

Using: cuda


In [6]:
#Rebuild Model architecture and load weights

class SiameseCNN(nn.Module):
    def __init__(self):
        super().__init__()
        base_model = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1)
        self.encoder = nn.Sequential(*list(base_model.children())[:-1])
        self.fc = nn.Sequential(
            nn.Linear(512, 256),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(256, 1),
            nn.Sigmoid()
        )

    def forward_once(self, x):
        out = self.encoder(x)
        return out.view(out.size(0), -1)

    def forward(self, img_left, img_right):
        feat_left  = self.forward_once(img_left)
        feat_right = self.forward_once(img_right)
        diff = feat_left - feat_right
        return self.fc(diff)

    def score(self, x):
        """Extract a single perception score for one image."""
        feat = self.forward_once(x)
        # Score = sigmoid of the norm of the embedding
        # Maps feature vector to a 0-1 wealth perception score
        return torch.sigmoid(feat.norm(dim=1, keepdim=True))

model = SiameseCNN().to(device)
model.load_state_dict(torch.load(MODEL_PATH, map_location=device))
model.eval()
print("Model loaded successfully.")

Model loaded successfully.


C:\Users\Aathira\AppData\Local\Temp\ipykernel_32312\1458365721.py:34: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model.load_state_dict(torch.load(MODEL_PATH, map_location

In [7]:
#Defining Transform and Scoring Dataset
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225])
])

# A simpler Dataset than PlacePulseDataset - I'm loading single images now,
# not pairs. I just need the image tensor and its filename (to extract LSOA
# code and coordinates from the filename for the output CSV).
class ScoringDataset(Dataset):
    """Dataset for scoring individual images — no pairs needed."""
    def __init__(self, image_dir, transform):
        self.image_dir = image_dir
        self.transform = transform
        self.filenames = [f for f in os.listdir(image_dir) if f.endswith('.jpg')]

    def __len__(self):
        return len(self.filenames)

    def __getitem__(self, idx):
        fname = self.filenames[idx]
        path  = os.path.join(self.image_dir, fname)
        img   = Image.open(path).convert('RGB')
        return self.transform(img), fname

dataset_2012 = ScoringDataset(IMG_2012, transform)
dataset_2019 = ScoringDataset(IMG_2019, transform)

loader_2012 = DataLoader(dataset_2012, batch_size=64, shuffle=False, num_workers=0)
loader_2019 = DataLoader(dataset_2019, batch_size=64, shuffle=False, num_workers=0)

print("2012 images to score:", len(dataset_2012))
print("2019 images to score:", len(dataset_2019))

2012 images to score: 6668
2019 images to score: 4612


In [8]:
# Load reference images from Place Pulse

# Loading 200 random Place Pulse images to use as scoring references.
# The scoring method is reference-based - each GM image is compared against
# these 200 references using the trained model, and its score = the proportion
# of those comparisons it wins (i.e. how often it looks wealthier).
# This converts the pairwise comparison model into a cardinal wealth score.
# np.random.seed(42) ensures the same 200 references are chosen every time.

PP_IMAGE_DIR = f'{base}\\Raw\\images_Chinese'

# Sample 200 random Place Pulse images as references
all_pp_images = [f for f in os.listdir(PP_IMAGE_DIR) if f.endswith('.jpg')]
np.random.seed(42)
ref_filenames = np.random.choice(all_pp_images, size=200, replace=False)

# Load and transform reference images into a tensor
ref_tensors = []
for fname in ref_filenames:
    path = os.path.join(PP_IMAGE_DIR, fname)
    try:
        img = Image.open(path).convert('RGB')
        ref_tensors.append(transform(img)) # Apply same transform as training
    except:
        pass # Skip any corrupted files silently

ref_tensor = torch.stack(ref_tensors).to(device)  # Stack into one tensor: (200, 3, 224, 224)
print("Reference images loaded:", ref_tensor.shape)

Reference images loaded: torch.Size([200, 3, 224, 224])


In [9]:
np.random.seed(20)
#Scoring Function
#For each GM Street View image:
# 1. Pass it through the ResNet encoder to get its 512-dim feature vector
# 2. Sample 50 random reference embeddings from the pre-computed 200
# 3. For each reference: compute (image_feat - ref_feat) and pass through the fc head
#    This gives the probability that the GM image looks wealthier than that reference
# 4. Average across all 50 comparisons - final wealth score between 0 and 1
# 5. Parse the LSOA code, lat, lon from the filename for the output CSV
def score_images(loader, year, ref_tensor, n_refs=50):
    results = []
    model.eval()

    # Pre-compute all 200 reference embeddings once - more efficient than recomputing them for every img
    with torch.no_grad():
        ref_feats = model.forward_once(ref_tensor)  # (200, 512)

    with torch.no_grad():
        for imgs, fnames in tqdm(loader, desc=f'Scoring {year}'):
            imgs = imgs.to(device)
            img_feats = model.forward_once(imgs)  # (batch, 512)

            batch_scores = []
            for i in range(len(imgs)):
                # Sample n_refs references randomly
                idx       = np.random.choice(len(ref_feats), size=n_refs, replace=False)
                refs_sub  = ref_feats[idx]           # (n_refs, 512)
                img_feat  = img_feats[i].unsqueeze(0) # (1, 512)

                # Compute diff: image - each reference
                diff      = img_feat - refs_sub       # (n_refs, 512)
                wins      = model.fc(diff)             # (n_refs, 1)
                score     = wins.mean().item()         # proportion won
                batch_scores.append(score)

            for fname, score in zip(fnames, batch_scores):
                # Filename format: E01004766_53.600965_-2.432860_2012.jpg
                parts    = fname.replace('.jpg', '').split('_')
                lsoa     = parts[0]
                lat      = float(parts[1])
                lon      = float(parts[2])
                results.append({
                    'filename':  fname,
                    'lsoa_code': lsoa,
                    'lat':       lat,
                    'lon':       lon,
                    'year':      year,
                    'score':     score
                })

    return pd.DataFrame(results)

In [10]:
#Running scoring on both years
print("Scoring 2012 images...")
scores_2012 = score_images(loader_2012, 2012, ref_tensor)
print(f"Done. Shape: {scores_2012.shape}")
print(scores_2012['score'].describe())

print("\nScoring 2019 images...")
scores_2019 = score_images(loader_2019, 2019, ref_tensor)
print(f"Done. Shape: {scores_2019.shape}")
print(scores_2019['score'].describe())

Scoring 2012 images...


Scoring 2012: 100%|██████████| 105/105 [02:28<00:00,  1.41s/it]


Done. Shape: (6668, 6)
count    6668.000000
mean        0.530982
std         0.104389
min         0.192893
25%         0.463040
50%         0.543187
75%         0.608870
max         0.786588
Name: score, dtype: float64

Scoring 2019 images...


Scoring 2019: 100%|██████████| 73/73 [01:42<00:00,  1.40s/it]

Done. Shape: (4612, 6)
count    4612.000000
mean        0.526124
std         0.106435
min         0.126018
25%         0.456660
50%         0.535156
75%         0.605424
max         0.806401
Name: score, dtype: float64


In [11]:
# Concatenating both years into one dataframe and saving as image_scores.csv.
# This is the image-level output — 20,018 rows, one per Street View image.
all_scores = pd.concat([scores_2012, scores_2019], ignore_index=True)
all_scores.to_csv(SCORES_OUT, index=False)
print("Image-level scores saved:", len(all_scores), "rows")
print(all_scores.head())

Image-level scores saved: 11280 rows
                                 filename  lsoa_code        lat       lon  \
0  E01004766_53.599887_-2.438451_2012.jpg  E01004766  53.599887 -2.438451   
1  E01004766_53.600965_-2.432860_2012.jpg  E01004766  53.600965 -2.432860   
2  E01004766_53.600968_-2.432851_2012.jpg  E01004766  53.600968 -2.432851   
3  E01004766_53.600976_-2.434356_2012.jpg  E01004766  53.600976 -2.434356   
4  E01004766_53.601197_-2.444864_2012.jpg  E01004766  53.601197 -2.444864   

   year     score  
0  2012  0.584896  
1  2012  0.247509  
2  2012  0.271680  
3  2012  0.552878  
4  2012  0.429897  


In [12]:
# Aggregating from image level to LSOA level.
# For each LSOA and year I compute the mean score across all approx 6 images.
# Then I pivot to wide format so each LSOA has one row with columns for
# mean_score_2012 and mean_score_2019.
# visual_change = mean_score_2019 - mean_score_2012 is my key independent variable.
# Positive = area looks more wealthy in 2019. Negative = visible deterioration.
lsoa_scores = (all_scores
               .groupby(['lsoa_code', 'year'])['score']
               .agg(['mean', 'std', 'count'])
               .reset_index())

lsoa_scores.columns = ['lsoa_code', 'year', 'mean_score', 'std_score', 'n_images']

# Pivot to wide format: one row per LSOA, separate columns for each year's mean score
lsoa_wide = lsoa_scores.pivot(index='lsoa_code', columns='year', values='mean_score').reset_index()
lsoa_wide.columns = ['lsoa_code', 'mean_score_2012', 'mean_score_2019']

# This is the variable that enters the regression as the independent variable
lsoa_wide['visual_change'] = lsoa_wide['mean_score_2019'] - lsoa_wide['mean_score_2012']

print("LSOA-level scores shape:", lsoa_wide.shape)
print(lsoa_wide.describe())
print("\nSample rows:")
print(lsoa_wide.head(10).to_string())

LSOA-level scores shape: (1627, 4)
       mean_score_2012  mean_score_2019  visual_change
count      1445.000000      1507.000000    1325.000000
mean          0.534035         0.525987      -0.006514
std           0.066040         0.076215       0.073107
min           0.300503         0.218863      -0.366715
25%           0.491252         0.476939      -0.050054
50%           0.536582         0.526496      -0.003563
75%           0.579013         0.578768       0.039050
max           0.742682         0.748147       0.252264

Sample rows:
   lsoa_code  mean_score_2012  mean_score_2019  visual_change
0  E01004766         0.417372         0.309407      -0.107966
1  E01004767         0.434464         0.487758       0.053294
2  E01004768         0.511049              NaN            NaN
3  E01004769         0.582074              NaN            NaN
4  E01004770         0.565633         0.605819       0.040186
5  E01004771         0.547276         0.458687      -0.088589
6  E01004772         0

In [13]:
LSOA_SCORES_OUT = f'{base}\\Outputs\\Scores\\lsoa_scores2.csv'
# Saving the LSOA-level scores to CSV and running sanity checks.
# Zero missing scores across 1,673 LSOAs = complete dataset, no attrition.
# Zero LSOAs with fewer than 3 images = all perception scores are based on
# sufficient observations to be reliable.

lsoa_wide.to_csv(LSOA_SCORES_OUT, index=False)
print("LSOA scores saved to:", LSOA_SCORES_OUT)

# Quick sanity checks
print("\nLSOAs with scores:", len(lsoa_wide))
print("LSOAs missing 2012 score:", lsoa_wide['mean_score_2012'].isna().sum())
print("LSOAs missing 2019 score:", lsoa_wide['mean_score_2019'].isna().sum())
print("\nvisual_change distribution:")
print(lsoa_wide['visual_change'].describe())

# If i get LSOAs with fewer than 3 images in either year — less reliable, so flagging
low_coverage = lsoa_scores[lsoa_scores['n_images'] < 3]
print(f"\nLSOAs with <3 images in at least one year: {len(low_coverage)}")

# Result: 0 - all LSOAs have sufficient image coverage for reliable scoring

LSOA scores saved to: E:\Warwick\EconDS\Dissertation\Outputs\Scores\lsoa_scores2.csv

LSOAs with scores: 1627
LSOAs missing 2012 score: 182
LSOAs missing 2019 score: 120

visual_change distribution:
count    1325.000000
mean       -0.006514
std         0.073107
min        -0.366715
25%        -0.050054
50%        -0.003563
75%         0.039050
max         0.252264
Name: visual_change, dtype: float64

LSOAs with <3 images in at least one year: 837
